# Google Earth Engine API

# Setup and Helper Functions

In [1]:
import math

import ee
import geemap

import pandas as pd
import csv
import time
import os
import warnings

/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [2]:
#!pip3 install earthengine-api
#!pip3 install geemap
#!pip3 install requests
#!pip3 install --upgrade requests urllib3 charset-normalizer

# LANDSAT and SENTINEL

**Resolution of Sentinel / Landsat (HLS)**  
Resolution: 30 meters × 30 meters  
Each pixel represents the average signal from a 30m square on Earth

**Filtering out T01 and T60**  
In HLS datasets, the system:index typically encodes the MGRS tile ID (Military Grid Reference System), which divides Earth into grid tiles.  

MGRS zones go from T01 to T60...correspond to specific longitudinal zones  
T01 and T60 sit at the edges of the coordinate system  

**These regions often:**  
 - wrap around longitude (near ±180°)  
 - create discontinuities  

**When combining tiles:**  
 - Neighboring tiles are usually consistent  
 - But edge tiles (01, 60) can:  
 - not align properly  
 - break mosaicking pipelines  

**In practice, those tiles often:**  
 - have less consistent coverage  
 - may include:  
  -* ocean-heavy regions  
  -* partial acquisitions  

So we are trying to avoid zones T01 and T60.

**system:index in HLS**  
For HLS datasets (HLSS30, HLSL30), the system:index looks roughly like:  
  HLS.S30.T10SEG.2020183T184919.v2.0  

**The key part "T10SEG" is the MGRS tile ID**  
T	Prefix (Tile)  
10	UTM Zone  
S	Latitude band  
EG	100km grid square  

**UTM Zone (01–60)**  
Earth is divided into 60 longitudinal strips  
Each zone = 6° longitude wide  

Examples:  
Zone 10: California region  
Zone 01: near 180°W  
Zone 60: near 180°E  

**Latitude Band (C–X, excluding I & O)**  
Horizontal strips (~8° tall)  

Example:  
S => roughly mid-latitudes (includes parts of US)  

**Grid Square (2 letters)**  
100 km × 100 km tile  

Example:  
EG => specific square inside zone 10S  

**With all that said:**  
T10SEG ==> Zone 10, Band S, Grid EG

## CODE: Extract Data

In [37]:
# search sentinel and landsat collections for data
def create_filters_GE(start_date, end_date, points, cloud_threshold):

    image_collection_sent = 'NASA/HLS/HLSS30/v002' # HLS Sentinel data
    image_collection_land = 'NASA/HLS/HLSL30/v002' # HLS Landsat data

    # T01 and T60 are anti-meridian zones whose bounding boxes wrap around
    # the globe and falsely match points in Central/South America via
    # filterBounds. Removing them is safe — our region (zones 14-22) uses
    # T14-T22 prefixes, which are never caught by these filters.
    antimeridian_filter = ee.Filter.And(
        ee.Filter.Not(ee.Filter.stringStartsWith('system:index', 'T01')),
        ee.Filter.Not(ee.Filter.stringStartsWith('system:index', 'T60'))
    )

    collection_sentinel = ee.ImageCollection(image_collection_sent)\
                            .filterDate(start_date, end_date)\
                            .filterBounds(points)\
                            .filter(ee.Filter.lt('CLOUD_COVERAGE', cloud_threshold))\
                            .filter(antimeridian_filter)

    collection_landsat = ee.ImageCollection(image_collection_land)\
                            .filterDate(start_date, end_date)\
                            .filterBounds(points)\
                            .filter(ee.Filter.lt('CLOUD_COVERAGE', cloud_threshold))\
                            .filter(antimeridian_filter)

    return collection_sentinel, collection_landsat

In [4]:
# Use fmask to remove invalid pixels with high cloud coverage
def create_image_mask(image, quality="medium"):
    fmask = image.select('Fmask')

    is_cirrus   = fmask.bitwiseAnd(1).neq(0)
    is_cloud    = fmask.bitwiseAnd(2).neq(0)
    is_adjacent = fmask.bitwiseAnd(4).neq(0)
    is_shadow   = fmask.bitwiseAnd(8).neq(0)

    if quality == "medium": # removes only direct cloud coverage
        mask = is_cirrus.Or(is_cloud).Not()
    if quality == "high": # removes clouds and shadows
        mask = is_cirrus.Or(is_cloud).Or(is_adjacent).Or(is_shadow).Not()

    return image.updateMask(mask)

In [5]:
# OPTION-1
def extract_median_values_at_point(image, points, buffer_meters, apply_mask=True):
    # Get time & cloud coverage
    time_start     = image.get('system:time_start')
    cloud_coverage = image.get('CLOUD_COVERAGE')

    # Create masked image, aka, removes cloudy.
    # NOTE: This affects which pixels are eligible for sampling
    #image_mask = create_image_mask(image)
    image_to_use = create_image_mask(image) if apply_mask else image

    # Create buffered zones for all points
    # Aka, Instead of looking at exactly that one spot,
    #      look at a small area around that spot.”
    # e.g. Give me values from a small circle around X (30 meters radius)”
    #      Landsat,Sentinel resolution is 30m x 30m rectangle.
    #      Our buffer involves a circle which covers 30m radius
    #      ontop of the current coordinates.
    #          ◯
    #        ( X )
    #          ◯
    # What are we trying to achieve?
    #   Instead of asking: “What is the temperature exactly at this GPS pin?”
    #     with buffer...
    #   We are asking: “What is the temperature in this small neighborhood?”
    buffered_points = points.map(
        lambda f: f.buffer(buffer_meters).copyProperties(f)
    )

    # Select desired bands for each sampled pixel in image
    bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12']

    # Compute median value inside each buffered area
    reduced = image_to_use.select(bands).reduceRegions(
        collection=buffered_points,
        reducer=ee.Reducer.median(),
        scale=30
    )

    # Add image metadata to each output feature
    def add_metadata(feature):
        return feature.set({
            'time': ee.Date(time_start).format('YYYY-MM-dd HH:mm:ss'),
            'CLOUD_COVERAGE': cloud_coverage,
            'row_id': feature.get('row_id'),
            'orig_lon': feature.get('orig_lon'),
            'orig_lat': feature.get('orig_lat')
        })

    return reduced.map(add_metadata)

In [6]:
# OPTION-2
def extract_values_at_point(image, points, buffer_meters):

    # Get time & cloud coverage
    time_start     = image.get('system:time_start')
    cloud_coverage = image.get('CLOUD_COVERAGE')

    # Create masked image, aka, removes cloudy.
    # NOTE: This affects which pixels are eligible for sampling
    image_mask = create_image_mask(image)

    # Create buffered zones for all points
    # Aka, Instead of looking at exactly that one spot,
    #      look at a small area around that spot.”
    # e.g. Give me values from a small circle around X (30 meters radius)”
    #      Landsat,Sentinel resolution is 30m x 30m rectangle.
    #      Our buffer involves a circle which covers 30m radius
    #      ontop of the current coordinates.
    #          ◯
    #        ( X )
    #          ◯
    # What are we trying to achieve?
    #   Instead of asking: “What is the temperature exactly at this GPS pin?”
    #     with buffer...
    #   We are asking: “What is the temperature in this small neighborhood?”
    buffered_geometry = points.map(lambda f: f.buffer(buffer_meters))

    # Select desired bands for each sampled pixel in image
    bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12']

    # Extract values at specified points & bands
    sampled_pixels = image_mask.select(bands).sampleRegions(
                                                    collection=buffered_geometry,
                                                    scale=30,
                                                    geometries=True)

    # Add time information: when this image was taken
    time_function  = lambda feature: feature.set('time',
                                                 ee.Date(time_start).format('YYYY-MM-dd HH:mm:ss'))

    # Add cloud information: cloud % of the entire image.
    cloud_function = lambda feature: feature.set('CLOUD_COVERAGE', cloud_coverage)

    #geometry_function = lambda feature: feature.setGeometry(feature.geometry())

    sampled_pixels = sampled_pixels.map(time_function).map(cloud_function)

    return sampled_pixels

## CODE: Feature engineering & create dataset

In [7]:
def get_data_list(filtered_images,
                  points,
                  max_elements,
                  buffer_meters,
                  apply_mask):
    # Extract information from each satellite image.
    n_images = filtered_images.size().getInfo()
    n_points = points.size().getInfo()

    print(f"Number of matching images: {n_images}")
    print(f"Number of input points: {n_points}")

    if n_images == 0 or n_points == 0:
        return {'features': []}

    # Keep flattened collection under the EE 5000-element abort threshold
    chunk_size = max(1, math.floor(max_elements / max(n_images, 1)))
    print(f"Using chunk size: {chunk_size}")

    all_features = []

    # Convert FeatureCollection to a list so we can slice it into chunks
    points_list = points.toList(n_points)

    for start in range(0, n_points, chunk_size):
        end = min(start + chunk_size, n_points)

        #print(f"Processing chunk: {start} to {end - 1}")

        point_chunk = ee.FeatureCollection(points_list.slice(start, end))

        collection_data = filtered_images.map(
            lambda image: extract_median_values_at_point(
                image,
                point_chunk,
                buffer_meters,
                apply_mask=apply_mask
            )
        ).flatten()

        chunk_info = collection_data.getInfo()
        all_features.extend(chunk_info.get('features', []))

    return {'features': all_features}

In [8]:
def safe_div(a, b):
    if a is None or b is None or (a + b) == 0:
        return None
    return (a - b) / (a + b)

### Code to create Sentinel dataframe

In [9]:
def create_sentinel_df(buffer_meters, filtered_images, points, max_elements, apply_mask):
    # Extract information from each satellite image.
    data_list_info = get_data_list(filtered_images,
                                   points,
                                   max_elements,
                                   buffer_meters,
                                   apply_mask)

    # Append data
    df_s30_data = []

    for feature in data_list_info['features']:
        props = feature['properties']
        #print("")
        #print(props)

        blue = props.get('B2')   # Blue band  - ~0.49 µm
        green = props.get('B3')   # Green band - ~0.56 µm
        red = props.get('B4')   # Red band   - ~0.665 µm

        # Sentinel red-edge bands
        red_edge1 = props.get('B5') # Red-edge 1 (~0.705 µm)
        red_edge2 = props.get('B6') # Red-edge 2 (~0.74 µm)
        red_edge3 = props.get('B7') # Red-edge 3 (~0.78 µm)

        nir = props.get('B8A')   # NIR   - Near Infrared band
        swir1 = props.get('B11') # SWIR1 - Shortwave Infrared Band 1
        swir2 = props.get('B12') # SWIR2 - Shortwave Infrared Band 2

        # Use original input coordinates, not centroid of buffer
        lon = props.get('orig_lon')
        lat = props.get('orig_lat')
        row_id = props.get('row_id')

        ndvi  = safe_div(nir, red)     #ndvi  = (nir - red) / (nir + red)
        mndwi = safe_div(green, swir1) #mndwi = (green - swir1) / (green + swir1)
        nbr   = safe_div(nir, swir2)   #nbr   = (nir - swir2) / (nir + swir2)
        if None in (nir, red, blue):
            evi = None
        else:
            denom = nir + 6 * red - 7.5 * blue + 1
            evi   = 2.5 * ((nir - red) / denom) if denom != 0 else None

        # Red-edge indices
        ndre1 = safe_div(nir, red_edge1)   # (NIR - RE1) / (NIR + RE1)
        ndre2 = safe_div(nir, red_edge2)   # (NIR - RE2) / (NIR + RE2)
        ndre3 = safe_div(nir, red_edge3)   # (NIR - RE3) / (NIR + RE3)

        if None in (nir, red_edge1) or red_edge1 == 0:
            ci_rededge = None
        else:
            ci_rededge = (nir / red_edge1) - 1

        # list of bands and variables to include in dataset
        df_s30_data.append({
            'time'  : props.get('time'),
            'row_id': props.get('row_id'),

            'longitude': lon,
            'latitude': lat,

            'Blue' : blue,
            'Green': green,
            'Red'  : red,

            'RE1': red_edge1,
            'RE2': red_edge2,
            'RE3': red_edge3,

            'NIR'  : nir,
            'SWIR1': swir1,
            'SWIR2': swir2,

            'NDVI' : ndvi,
            'MNDWI': mndwi,
            'NBR'  : nbr,
            'EVI'  : evi,

            'NDRE1': ndre1,
            'NDRE2': ndre2,
            'NDRE3': ndre3,
            'CIrededge': ci_rededge,

            'CLOUD_COVERAGE': props.get('CLOUD_COVERAGE')
        })

    print("")
    return pd.DataFrame(df_s30_data)

### Code to find medain of values between multiple images.

In [10]:
#OPTION-1: If there are multiple images, reduce them by finding the medain between them.
def reduce_df_by_median_across_images(df):
    expected_cols = [
        'row_id', 'longitude', 'latitude', 'time',
        'Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3',
        'NIR', 'SWIR1', 'SWIR2',
        'NDVI', 'MNDWI', 'NBR', 'EVI',
        'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge',
        'CLOUD_COVERAGE'
    ]
    band_cols = [
        'Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2',
        'RE1', 'RE2', 'RE3',
        'NDVI', 'MNDWI', 'NBR', 'EVI',
        'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge',
        'CLOUD_COVERAGE'
    ]
    numeric_cols = [col for col in band_cols if col in df.columns]

    # Must come AFTER expected_cols is defined
    # Also catches phantom T01/T60 rows where all band values are null
    if df.empty or df[band_cols].isnull().all(axis=None):
        return pd.DataFrame(columns=expected_cols)

    # Drop rows where every band is null
    df = df.dropna(subset=band_cols, how='all')
    if df.empty:
        return pd.DataFrame(columns=expected_cols)

    point_col = 'row_id'
    grouped   = df.groupby(point_col, as_index=False)

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='Mean of empty slice',
                                category=RuntimeWarning)
        warnings.filterwarnings('ignore', message='Degrees of freedom <= 0',
                                category=RuntimeWarning)
        df_median = grouped[numeric_cols].median()

    coords_df = grouped[['longitude', 'latitude']].first()
    time_df   = grouped['time'].first()
    df_final  = df_median.merge(coords_df, on=point_col, how='left')
    df_final  = df_final.merge(time_df,    on=point_col, how='left')

    preferred_order = [
        point_col, 'longitude', 'latitude', 'time',
        'Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3',
        'NIR', 'SWIR1', 'SWIR2',
        'NDVI', 'MNDWI', 'NBR', 'EVI',
        'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge',
        'CLOUD_COVERAGE'
    ]
    ordered_cols   = [c for c in preferred_order if c in df_final.columns]
    remaining_cols = [c for c in df_final.columns if c not in preferred_order]
    return df_final[ordered_cols + remaining_cols]

### Code to select the image with least cloudiness

In [11]:
# OPTION-2: If there are multiple images, reduce them by finding the least cloudy one.

def reduce_df_by_least_cloudy_image(df):
    if df.empty:
        return df.copy()

    point_col = 'row_id' if 'row_id' in df.columns else 'row_id'

    # Prefer rows that actually have band data
    # Here I use Blue as a proxy for "valid data exists"
    valid_df = df[df['Blue'].notna()].copy()

    # If nothing valid exists, fall back to original df
    if valid_df.empty:
        valid_df = df.copy()

    # Sort so least cloudy row comes first for each point
    valid_df = valid_df.sort_values(
        by=[point_col, 'CLOUD_COVERAGE'],
        ascending=[True, True]
    )

    df_final = valid_df.groupby(point_col, as_index=False).first()

    preferred_order = [
        point_col, 'longitude', 'latitude', 'time',
        'Blue', 'Green', 'Red',
        'RE1', 'RE2', 'RE3',
        'NIR', 'SWIR1', 'SWIR2',
        'NDVI', 'MNDWI', 'NBR', 'EVI',
        'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge',
        'CLOUD_COVERAGE'
    ]
    ordered_cols   = [c for c in preferred_order if c in df_final.columns]
    remaining_cols = [c for c in df_final.columns if c not in preferred_order]

    return df_final[ordered_cols + remaining_cols]

# PROCESS AGB DATA

In [12]:
def get_agb_data(csv_file):
    orig_df = pd.read_csv(csv_file)

    orig_df['start_date'] = pd.to_datetime(orig_df['start_date'], errors='coerce')
    orig_df['end_date']   = pd.to_datetime(orig_df['end_date'],   errors='coerce')

    sentinel_threshold_start = pd.Timestamp('2017-01-01')
    sentinel_threshold_end   = pd.Timestamp('2017-12-31')

    agb_df = orig_df.copy()

    # Preserve original survey dates
    agb_df['capture_start'] = agb_df['start_date']
    agb_df['capture_end']   = agb_df['end_date']

    # Clamp capture_start: if before 2017-01-01, set to 2017-01-01
    early_start_mask = agb_df['capture_start'] < sentinel_threshold_start
    agb_df.loc[early_start_mask, 'capture_start'] = sentinel_threshold_start

    # Clamp capture_end: if before 2017-12-31, set to 2017-12-31
    early_end_mask = agb_df['capture_end'] < sentinel_threshold_end
    agb_df.loc[early_end_mask, 'capture_end'] = sentinel_threshold_end

    # Format capture dates as strings for GEE filterDate
    agb_df['capture_start'] = agb_df['capture_start'].dt.strftime('%Y-%m-%d')
    agb_df['capture_end']   = agb_df['capture_end'].dt.strftime('%Y-%m-%d')

    # Keep original dates as readable strings
    agb_df['start_date'] = agb_df['start_date'].dt.strftime('%Y-%m-%d')
    agb_df['end_date']   = agb_df['end_date'].dt.strftime('%Y-%m-%d')

    return agb_df

In [13]:
def build_points_from_group(group_df):
    point_features = []

    for _, row in group_df.iterrows():
        lon = pd.to_numeric(row['longitude'], errors='coerce')
        lat = pd.to_numeric(row['latitude'], errors='coerce')

        row_id = int(row['row_id'])

        if pd.isna(lon) or pd.isna(lat):
            print(f"Skipping invalid coordinate: row_id={row_id}, lon={lon}, lat={lat}")
            continue

        point_features.append(
            ee.Feature(ee.Geometry.Point([lon, lat])).set({
                'row_id': row_id,
                'orig_lon': lon,
                'orig_lat': lat
            })
        )
    print(f"Valid points in group: {len(point_features)} / {len(group_df)}")

    return ee.FeatureCollection(point_features)

In [42]:
def process_date_group_sentinel_only(df_group, buffer_meters, max_elements,
                                     cloud_threshold=10, fallback_threshold=50,
                                     apply_mask=True):
    start_date = str(df_group['capture_start'].iloc[0])
    end_date   = str(df_group['capture_end'].iloc[0])
    points     = build_points_from_group(df_group)

    # Pass 1: strict tile-level threshold
    filter_sentinel, _ = create_filters_GE(start_date, end_date, points,
                                           cloud_threshold=cloud_threshold)
    
    df_s30       = create_sentinel_df(buffer_meters, filter_sentinel, points,
                                      max_elements, apply_mask)
    df_s30_final = reduce_df_by_median_across_images(df_s30)

    base_df      = pd.DataFrame({'row_id': df_group['row_id'].astype(int)})
    df_s30_final = base_df.merge(df_s30_final, on='row_id', how='left')
    df_s30_final['cloud_threshold_used'] = cloud_threshold

    # Pass 2: fallback for persistently cloudy regions
    null_mask    = df_s30_final['Blue'].isnull()
    null_row_ids = df_s30_final.loc[null_mask, 'row_id'].tolist()

    if null_row_ids:
        print(f"  Pass 1: {(~null_mask).sum()} points at {cloud_threshold}% threshold")
        print(f"  Pass 2: {len(null_row_ids)} null points retrying "
              f"at {fallback_threshold}% threshold")

        fallback_group  = df_group[df_group['row_id'].isin(null_row_ids)]
        fallback_points = build_points_from_group(fallback_group)

        filter_fallback, _ = create_filters_GE(start_date, end_date,
                                               fallback_points,
                                               cloud_threshold=fallback_threshold)
        df_fb       = create_sentinel_df(buffer_meters, filter_fallback,
                                         fallback_points, max_elements, apply_mask)
        df_fb_final = reduce_df_by_median_across_images(df_fb)

        fb_base     = pd.DataFrame({'row_id': fallback_group['row_id'].astype(int)})
        df_fb_final = fb_base.merge(df_fb_final, on='row_id', how='left')
        df_fb_final['cloud_threshold_used'] = fallback_threshold

        still_null = df_fb_final['Blue'].isnull().sum()
        if still_null > 0:
            print(f"  After pass 2: {still_null} points still null "
                  f"— genuinely no imagery available")

        # Exclude empty or all-NA frames before concat to avoid FutureWarning
        # triggered when pass 1 yields zero rows (all points went to fallback)
        pass1_df = df_s30_final[~null_mask].copy()
        frames   = [f for f in [pass1_df, df_fb_final]
                    if not f.empty and not f.dropna(how='all').empty]

        df_s30_final = pd.concat(frames, ignore_index=True) \
                       if frames else pd.DataFrame(columns=df_s30_final.columns)

    df_s30_final = df_s30_final.rename(columns={'time': 'sentinel_time'})
    df_s30_final = df_s30_final.drop(columns=['latitude', 'longitude'],
                                     errors='ignore')
    merged = df_group.merge(df_s30_final, on='row_id', how='left')
    return merged

In [15]:
def split_dataframe_into_chunks(df, chunk_size):
    chunks = []
    n = len(df)

    for start in range(0, n, chunk_size):
        end = start + chunk_size
        chunks.append(df.iloc[start:end].copy())

    return chunks

In [16]:
def run(input_csv, max_elements, buffer_meters, cloud_threshold):
    df = get_agb_data(input_csv)
    df = df.reset_index(drop=True)
    df['row_id']    = df.index.astype(int)
    df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
    df = df.dropna(subset=['latitude', 'longitude'])

    results      = []
    #grouped      = df.groupby(['capture_start', 'capture_end'], sort=False)
    grouped      = df.groupby(['dataset', 'capture_start', 'capture_end'], sort=False)
    total_groups = len(grouped)

    for group_num, ((dataset, start_date, end_date), df_group) in enumerate(grouped, start=1):
        print(
            f'Processing group {group_num}/{total_groups} | '
            f'dataset={dataset} | '
            f'capture={start_date} → {end_date} | '
            f'rows={len(df_group)}'
        )
        merged_group = process_date_group_sentinel_only(df_group,
                                                        buffer_meters=buffer_meters,
                                                        max_elements=max_elements,
                                                        cloud_threshold=cloud_threshold)
        results.append(merged_group)

    df_out = pd.concat(results, ignore_index=True)
    df_out = df_out.sort_values('row_id').drop(columns=['row_id'])
    return df_out

# Authenticate GCP session.

In [17]:
#project_id = "gen-lang-client-0271313824"
project_id = "ac215-bhargav"

ee.Authenticate()
ee.Initialize(project=project_id)

# USER DEFINED VARIABLES

In [18]:
# For testing only
#INPUT_CSV  = "../DATA/AGB_DATA/TEST_AGB.csv"
#OUTPUT_CSV = "TEST_EO.csv"

BASE = "../../DATA/AGB_DATA/Merged_Data/"

# Actual files
INPUT_CSV_MAIN     = BASE + "AGB_MERGED.csv"
INPUT_CSV_VALIDATE = BASE + "AGB_VALIDATE.csv"

# Sentinel data only
OUTPUT_CSV_MAIN_SENTINEL = BASE + "Sentinel_AGB/AGB_EO_SENTINEL.csv"
OUTPUT_CSV_VAL_SENTINEL  = BASE + "Sentinel_AGB/AGB_VAL_EO_SENTINEL.csv"

# EXTRACT SENTINEL DATA

## Parameters

In [19]:
# If a 30 m buffer is used: A 30 m/pixel image will give roughly 1 pixel of coverage per point.
# After create_image_mask masks cloudy pixels, if that single pixel is masked, 
# ee.Reducer.median() has nothing to work with and returns null for every band at that point. 
buffer_meters = 90 # covers ~3x3 pixel neighborhood

In [20]:
max_elements  = 4500

In [21]:
cloud_threshold = 10

## Execution

In [44]:
start = time.time()
main_df = run(INPUT_CSV_MAIN, max_elements, buffer_meters, cloud_threshold)
end = time.time()
print(f"Time taken to work on main csv file: {end - start:.2f} seconds")
print("")

Processing group 1/17 | dataset=ElSalvador | capture=2017-01-01 → 2017-12-31 | rows=2857
Valid points in group: 2857 / 2857
Number of matching images: 22
Number of input points: 2857
Using chunk size: 204

Processing group 2/17 | dataset=Panama-Chirqui_2 | capture=2017-01-01 → 2017-12-31 | rows=83
Valid points in group: 83 / 83
Number of matching images: 0
Number of input points: 83

  Pass 1: 0 points at 10% threshold
  Pass 2: 83 null points retrying at 50% threshold
Valid points in group: 83 / 83
Number of matching images: 8
Number of input points: 83
Using chunk size: 562

Processing group 3/17 | dataset=CostaRica-Nicoya | capture=2017-01-01 → 2017-12-31 | rows=313
Valid points in group: 313 / 313
Number of matching images: 9
Number of input points: 313
Using chunk size: 500

  Pass 1: 203 points at 10% threshold
  Pass 2: 110 null points retrying at 50% threshold
Valid points in group: 110 / 110
Number of matching images: 18
Number of input points: 110
Using chunk size: 250

Proce

In [47]:
main_df.columns

Index(['dataset', 'plot_id', 'start_date', 'end_date', 'latitude', 'longitude',
       'diameter', 'height', 'species', 'plant_AGB_kg', 'capture_start',
       'capture_end', 'sentinel_time', 'Blue', 'Green', 'Red', 'RE1', 'RE2',
       'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1',
       'NDRE2', 'NDRE3', 'CIrededge', 'CLOUD_COVERAGE',
       'cloud_threshold_used'],
      dtype='object')

## Audit the generated data

### Cloud threshold debugger

Diagnoses missing band values for a single dataset by inspecting GEE tile
coverage, comparing masked vs unmasked extraction, and running the two-pass
cloud threshold fallback to confirm all points can be resolved.

Diagnoses why a dataset returns NULL band values from the HLS
Sentinel-2 pipeline by running three sequential checks:

1. GEE Image Inspection
   Confirms how many tiles pass the cloud threshold, verifies the
   tile actually contains pixels at the point location, and shows
   raw reduceRegions output with and without buffer.

2. Mask Comparison
   Runs extraction with Fmask ON and OFF to determine whether nulls
   are caused by pixel-level cloud masking or by missing imagery.

3. Two-Pass Extraction
   Runs the full production pipeline with strict threshold first,
   then retries null points at the fallback threshold. Reports final
   null count and which threshold resolved each point.

In [66]:
def diagnose_dataset_extraction(dataset_name, raw_df, cloud_threshold=10,
                                 fallback_threshold=50, buffer_meters=90,
                                 max_elements=4500):
    """
    Full diagnostic for a single dataset — runs GEE inspection, mask
    comparison, and two-pass extraction to isolate exactly where nulls
    come from.

    Parameters
    ----------
    dataset_name       : str   — must match a value in raw_df['dataset']
    raw_df             : df    — output of get_agb_data with row_id assigned
    cloud_threshold    : int   — pass 1 tile-level cloud threshold
    fallback_threshold : int   — pass 2 fallback threshold
    buffer_meters      : int   — buffer radius for reduceRegions
    max_elements       : int   — EE element limit for chunking
    """

    BAND_COLS = [
        'Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3',
        'NIR', 'SWIR1', 'SWIR2'
    ]

    def count_nulls(df):
        available = [c for c in BAND_COLS if c in df.columns]
        if not available:
            return 'no band columns found'
        return int(df[available].isnull().any(axis=1).sum())

    print("=" * 60)
    print(f"DATASET DIAGNOSTIC: {dataset_name}")
    print("=" * 60)

    # ── Setup ─────────────────────────────────────────────────────────────────
    test_group = raw_df[raw_df['dataset'] == dataset_name].copy()
    if test_group.empty:
        print(f"ERROR: dataset '{dataset_name}' not found in raw_df")
        return None

    capture_start = test_group['capture_start'].iloc[0]
    capture_end   = test_group['capture_end'].iloc[0]
    print(f"Rows          : {len(test_group)}")
    print(f"capture_start : {capture_start}")
    print(f"capture_end   : {capture_end}")
    print(f"Bands checked : {BAND_COLS}")

    points = build_points_from_group(test_group)

    # ── GEE image inspection ──────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print("GEE IMAGE INSPECTION")
    print(f"{'─'*60}")

    filter_sentinel, _ = create_filters_GE(capture_start, capture_end,
                                           points, cloud_threshold)
    n_images = filter_sentinel.size().getInfo()
    print(f"Images at {cloud_threshold}% threshold : {n_images}")

    if n_images > 0:
        single_image = ee.Image(filter_sentinel.first())
        single_point = ee.FeatureCollection([ee.Feature(points.first())])
        image_info   = single_image.getInfo()

        print(f"First image ID : {image_info['id']}")
        print(f"Band names     : {[b['id'] for b in image_info['bands']]}")
        print(f"Cloud cover    : {single_image.get('CLOUD_COVERAGE').getInfo()}")

        point_info = single_point.first().getInfo()
        print(f"Point props    : {point_info['properties']}")
        print(f"Point geom     : {point_info['geometry']}")

        print("\n--- reduceRegions (no mask, no buffer) ---")
        raw_reduced = single_image.reduceRegions(
            collection=single_point,
            reducer=ee.Reducer.median(),
            scale=30
        )
        raw_info = raw_reduced.getInfo()
        print(f"Feature count  : {len(raw_info['features'])}")
        if raw_info['features']:
            print(f"Properties     : {raw_info['features'][0]['properties']}")

        print(f"\n--- reduceRegions (no mask, {buffer_meters}m buffer) ---")
        buffered = single_point.map(
            lambda f: f.buffer(buffer_meters).copyProperties(f)
        )
        buf_reduced = single_image.reduceRegions(
            collection=buffered,
            reducer=ee.Reducer.median(),
            scale=30
        )
        buf_info = buf_reduced.getInfo()
        print(f"Feature count  : {len(buf_info['features'])}")
        if buf_info['features']:
            print(f"Properties     : {buf_info['features'][0]['properties']}")
    else:
        print(f"No images at {cloud_threshold}% — skipping GEE inspection "
              f"(will be handled by fallback)")

    # ── Mask comparison ───────────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print("MASK COMPARISON")
    print(f"{'─'*60}")

    for label, apply_mask in [("With pixel mask (Fmask)", True),
                               ("Without pixel mask",      False)]:
        result  = process_date_group_sentinel_only(test_group,
                                                   buffer_meters=buffer_meters,
                                                   max_elements=max_elements,
                                                   cloud_threshold=cloud_threshold,
                                                   fallback_threshold=fallback_threshold,
                                                   apply_mask=apply_mask)
        null_count = count_nulls(result)
        print(f"  {label:30s} → Null rows: {null_count} / {len(result)}")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print("SUMMARY")
    print(f"{'─'*60}")

    null_count = count_nulls(result)
    status     = "PASS" if null_count == 0 else "FAIL"
    print(f"\n{status} — Null rows: {null_count} / {len(result)}")

    if 'cloud_threshold_used' in result.columns:
        print("Threshold distribution:")
        for threshold, count in result['cloud_threshold_used']\
                                       .value_counts().items():
            print(f"  threshold={threshold}% : {count} points")

    return result

In [67]:
raw_df = get_agb_data(INPUT_CSV_MAIN)
raw_df = raw_df.reset_index(drop=True)
raw_df['row_id']    = raw_df.index.astype(int)
raw_df['latitude']  = pd.to_numeric(raw_df['latitude'],  errors='coerce')
raw_df['longitude'] = pd.to_numeric(raw_df['longitude'], errors='coerce')
raw_df = raw_df.dropna(subset=['latitude', 'longitude'])

In [68]:
# SINGLE FAILING DATASET
result = diagnose_dataset_extraction('Panama-Chirqui_2', raw_df)

DATASET DIAGNOSTIC: Panama-Chirqui_2
Rows          : 83
capture_start : 2017-01-01
capture_end   : 2017-12-31
Bands checked : ['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2']
Valid points in group: 83 / 83

────────────────────────────────────────────────────────────
GEE IMAGE INSPECTION
────────────────────────────────────────────────────────────
Images at 10% threshold : 0
No images at 10% — skipping GEE inspection (will be handled by fallback)

────────────────────────────────────────────────────────────
MASK COMPARISON
────────────────────────────────────────────────────────────
Valid points in group: 83 / 83
Number of matching images: 0
Number of input points: 83

  Pass 1: 0 points at 10% threshold
  Pass 2: 83 null points retrying at 50% threshold
Valid points in group: 83 / 83
Number of matching images: 8
Number of input points: 83
Using chunk size: 562

  With pixel mask (Fmask)        → Null rows: 0 / 83
Valid points in group: 83 / 83
Number of matching 

#### Intepretation of the result

**RESULT-1: GEE Image Inspection**  
**OUTPUT**  
Images at 10% threshold : 0  
No images at 10% — skipping GEE inspection  
**COMMENT:**  
 - No Sentinel-2 tiles passed the strict 10% cloud filter for Panama-Chirqui_2 in 2017
 - The entire tile is too cloudy on every acquisition day
 - Nothing to inspect

**RESULT-2: Mask Comparison**  
**OUTPUT**  
Pass 1: 0 points at 10% threshold  
Pass 2: 83 null points retrying at 50% threshold  
Number of matching images: 8  

With pixel mask (Fmask)  → Null rows: 0 / 83  
Without pixel mask       → Null rows: 0 / 83  
**COMMENT:**  
 - Pass 1 found nothing at 10% cloud threshold
 - Pass 2 found 8 images at the relaxed 50% threshold
 - Both masked and unmasked return 0 nulls (0 null rows after Fmask pixel-level cleaning)
 - Fmask is not the problem here, the 8 images have enough clean pixels regardless of masking
 - Fmask made no difference → the pixels at your measurement locations happen to be cloud-free even on days when 40-50% of the tile is cloudy

This is acceptable. Tile-wide CLOUD_COVERAGE is a coarse metric — it reflects the whole 109.8km × 109.8km MGRS tile.  
The mangrove plots are small, localized, and could easily sit in the clear portion of a partially cloudy tile.  
Fmask confirms the pixels are clean.

**RESULT-3: SUMMARY**  
threshold=50% : 83 points  
**COMMENT:**  
 - All 83 points successfully extracted.
 - Every single point needed the fallback threshold — this dataset lives in a persistently cloudy coastal rainforest region where no image ever has tile-wide cloud cover below 10%.

#### DIAGNOSE NULLs

In [58]:
def diagnose_nulls(df):
    band_cols = [
        'Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3',
        'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR',
        'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge'
    ]

    null_mask = df[band_cols].isnull().any(axis=1)
    null_df   = df[null_mask]
    ok_df     = df[~null_mask]
    total     = len(df)

    if not len(null_df):
        print("There is no NULL data")
        return

    # OVERVIEW
    print("=" * 60)
    print("OVERVIEW")
    print("=" * 60)
    print(f"Total rows             : {total}")
    print(f"Rows with any NULL     : {len(null_df)}  ({100*len(null_df)/total:.1f}%)")
    print(f"Rows fully OK          : {len(ok_df)}  ({100*len(ok_df)/total:.1f}%)")
    print(f"Total NULL cell count  : {df[band_cols].isnull().sum().sum()}")

    # NULL count per column
    print("\n" + "=" * 60)
    print("NULL COUNT PER COLUMN (affected rows only)")
    print("=" * 60)
    null_col_counts = null_df[band_cols].isnull().sum().sort_values(ascending=False)
    print(null_col_counts[null_col_counts > 0].to_string())

    # Geographic spread
    print("\n" + "=" * 60)
    print("GEOGRAPHIC SPREAD")
    print("=" * 60)
    print("NULL rows:")
    print(null_df[['latitude', 'longitude']].describe().to_string())
    print("\nOK rows:")
    print(ok_df[['latitude', 'longitude']].describe().to_string())
    print("\nSample NULL coordinates (sorted by longitude):")
    print(null_df[['dataset', 'latitude', 'longitude']]
          .sort_values('longitude').head(20).to_string(index=False))

    # Date-wise analysis
    print("\n" + "=" * 60)
    print("DATE ANALYSIS")
    print("=" * 60)
    print("Top start_date values in NULL rows:")
    print(null_df['start_date'].value_counts().head(10).to_string())

    df_tmp = df.copy()
    df_tmp['date_window_days'] = (
        pd.to_datetime(df_tmp['end_date']) - pd.to_datetime(df_tmp['start_date'])
    ).dt.days
    print("\nDate window length — NULL rows:")
    print(df_tmp[null_mask]['date_window_days'].describe().to_string())
    print("\nDate window length — OK rows:")
    print(df_tmp[~null_mask]['date_window_days'].describe().to_string())

    # Per-dataset breakdown
    print("\n" + "=" * 60)
    print("PER-DATASET BREAKDOWN")
    print("=" * 60)
    summary = []
    for dataset, group in df.groupby('dataset'):
        n_total  = len(group)
        n_null   = group[band_cols].isnull().any(axis=1).sum()
        n_ok     = n_total - n_null
        pct_null = 100 * n_null / n_total
        summary.append({
            'dataset'   : dataset,
            'total'     : n_total,
            'ok'        : n_ok,
            'null'      : n_null,
            'null_%'    : round(pct_null, 1),
            'mean_lat'  : round(group['latitude'].mean(), 3),
            'mean_lon'  : round(group['longitude'].mean(), 3),
            'date_range': f"{group['start_date'].min()} → {group['start_date'].max()}",
        })
    summary_df = pd.DataFrame(summary).sort_values('null_%', ascending=False)
    print(summary_df.to_string(index=False))

    # Fully missing datasets
    fully_missing = summary_df[summary_df['null_%'] == 100.0]
    if not fully_missing.empty:
        print("\n" + "=" * 60)
        print("DATASETS WITH NO IMAGERY AT ALL (100% null)")
        print("=" * 60)
        print(fully_missing[['dataset', 'total', 'mean_lat', 'mean_lon', 'date_range']]
              .to_string(index=False))

    # Partially missing datasets
    partially_missing = summary_df[
        (summary_df['null_%'] > 0) & (summary_df['null_%'] < 100.0)
    ]
    if not partially_missing.empty:
        print("\n" + "=" * 60)
        print("DATASETS WITH PARTIAL NULLS")
        print("=" * 60)
        print(partially_missing[['dataset', 'total', 'ok', 'null', 'null_%',
                                  'mean_lat', 'mean_lon']].to_string(index=False))

    # Capture window for null rows
    if 'capture_start' in df.columns and 'capture_end' in df.columns:
        print("\n" + "=" * 60)
        print("CAPTURE DATE WINDOWS FOR NULL ROWS")
        print("=" * 60)
        print(null_df.groupby(['dataset', 'capture_start', 'capture_end'])
                     .size()
                     .reset_index(name='null_count')
                     .sort_values('null_count', ascending=False)
                     .to_string(index=False))

    return summary_df

In [59]:
summary = diagnose_nulls(main_df)

There is no NULL data


### DIAGNOSE TILE COVERAGE

In [60]:
def diagnose_tile_coverage(dataset_name, lon, lat, start_date, end_date,
                           cloud_threshold=10,
                           bbox_buffer_deg=3,
                           region_boxes=None):
    """
    Diagnoses why a dataset location returns null values from HLS Sentinel-2.
    Checks four things:
      1. What tiles filterBounds(point) returns
      2. What survives after removing T01/T60 anti-meridian phantoms
      3. What real tiles exist when using a bounding box instead of a point
      4. Regional coverage check across named bounding boxes

    Parameters
    ----------
    dataset_name    : str            — label for print output
    lon, lat        : float          — representative coordinate from the dataset
    start_date      : str            — 'YYYY-MM-DD'
    end_date        : str            — 'YYYY-MM-DD'
    cloud_threshold : int            — CLOUD_COVERAGE upper limit (default 10)
    bbox_buffer_deg : float          — degrees to expand around point for bbox search
    region_boxes    : dict, optional — {region_name: ee.Geometry.BBox}
                                       if None, auto-generates one from the point

    Usage
    -----
    # Single dataset, auto bbox
    diagnose_tile_coverage('Panama-Chirqui_2', lon=-82.28, lat=8.31,
                           start_date='2017-01-01', end_date='2017-12-31')

    # Single dataset, named region boxes
    region_boxes = {
        'Panama'       : ee.Geometry.BBox(-84,  7, -77, 10),
        'CostaRica'    : ee.Geometry.BBox(-86,  8, -82, 11),
        'Brazil-Amazon': ee.Geometry.BBox(-48, -2, -46,  0),
    }
    diagnose_tile_coverage('Panama-Chirqui_2', lon=-82.28, lat=8.31,
                           start_date='2017-01-01', end_date='2017-12-31',
                           region_boxes=region_boxes)

    # All failing datasets at once
    summary = diagnose_nulls(main_df)
    failing = summary[summary['null_%'] > 0]
    for _, row in failing.iterrows():
        sample = main_df[main_df['dataset'] == row['dataset']].iloc[0]
        diagnose_tile_coverage(
            dataset_name = row['dataset'],
            lon          = sample['longitude'],
            lat          = sample['latitude'],
            start_date   = sample['capture_start'],
            end_date     = sample['capture_end'],
            region_boxes = region_boxes
        )
    """
    from collections import Counter

    print("=" * 60)
    print(f"TILE DIAGNOSTIC: {dataset_name}")
    print(f"  Point          : ({lat}, {lon})")
    print(f"  Date range     : {start_date} → {end_date}")
    print(f"  Cloud threshold: < {cloud_threshold}%")
    print("=" * 60)

    def get_tile(image):
        tile = ee.String(image.get('system:index')).slice(0, 6)
        return image.set('tile', tile)

    antimeridian_filter = ee.Filter.And(
        ee.Filter.Not(ee.Filter.stringStartsWith('system:index', 'T01')),
        ee.Filter.Not(ee.Filter.stringStartsWith('system:index', 'T60'))
    )

    point = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lon, lat]))])
    bbox  = ee.Geometry.BBox(
        lon - bbox_buffer_deg, lat - bbox_buffer_deg,
        lon + bbox_buffer_deg, lat + bbox_buffer_deg
    )

    # --- Step 1: filterBounds(point) raw ---
    col_point = ee.ImageCollection('NASA/HLS/HLSS30/v002')\
                    .filterDate(start_date, end_date)\
                    .filterBounds(point)\
                    .filter(ee.Filter.lt('CLOUD_COVERAGE', cloud_threshold))
    n_point    = col_point.size().getInfo()
    tiles_point = Counter(
        col_point.map(get_tile).aggregate_array('tile').getInfo()
    )
    print(f"\n[1] filterBounds(point) — {n_point} images")
    if tiles_point:
        for tile, count in sorted(tiles_point.items(), key=lambda x: -x[1]):
            print(f"    {tile} : {count}")
    else:
        print("    No images found")

    # --- Step 2: filterBounds(point) after T01/T60 removal ---
    col_filtered   = col_point.filter(antimeridian_filter)
    n_filtered     = col_filtered.size().getInfo()
    tiles_filtered = Counter(
        col_filtered.map(get_tile).aggregate_array('tile').getInfo()
    )
    print(f"\n[2] After T01/T60 filter — {n_filtered} images")
    if tiles_filtered:
        for tile, count in sorted(tiles_filtered.items(), key=lambda x: -x[1]):
            print(f"    {tile} : {count}")
    else:
        print("    No images found — filterBounds(point) broken for this location")

    # --- Step 3: filterBounds(bbox) after T01/T60 removal ---
    col_bbox   = ee.ImageCollection('NASA/HLS/HLSS30/v002')\
                    .filterDate(start_date, end_date)\
                    .filterBounds(bbox)\
                    .filter(ee.Filter.lt('CLOUD_COVERAGE', cloud_threshold))\
                    .filter(antimeridian_filter)
    n_bbox     = col_bbox.size().getInfo()
    tiles_bbox = Counter(
        col_bbox.map(get_tile).aggregate_array('tile').getInfo()
    )
    utm_zones  = set(t[:3] for t in tiles_bbox)
    print(f"\n[3] filterBounds(bbox ±{bbox_buffer_deg}°) after T01/T60 filter"
          f" — {n_bbox} images | UTM zones: {utm_zones}")
    if tiles_bbox:
        for tile, count in sorted(tiles_bbox.items(), key=lambda x: -x[1]):
            print(f"    {tile} : {count}")
    else:
        print("    No images found even with bbox — data genuinely unavailable")

    # --- Step 4: named region boxes ---
    if region_boxes is None:
        # Auto-generate a single region box from the point
        region_boxes = {dataset_name: bbox}

    print(f"\n[4] Regional coverage check")
    for region, rbbox in region_boxes.items():
        col_region   = ee.ImageCollection('NASA/HLS/HLSS30/v002')\
                        .filterDate(start_date, end_date)\
                        .filterBounds(rbbox)\
                        .filter(ee.Filter.lt('CLOUD_COVERAGE', cloud_threshold))\
                        .filter(antimeridian_filter)
        n_region     = col_region.size().getInfo()
        tiles_region = col_region.map(get_tile).aggregate_array('tile').getInfo()
        utm_zones_r  = set(t[:3] for t in tiles_region)
        status       = "✅" if n_region > 0 else "❌"
        print(f"    {status} {region:20s} : {n_region:>3} images "
              f"| UTM zones: {utm_zones_r}")

    # --- Verdict ---
    print(f"\n{'─' * 60}")
    if n_point > 0 and n_filtered == 0 and n_bbox > 0:
        print("VERDICT: Anti-meridian phantom tiles + filterBounds(point) broken")
        print("  FIX   : Use filterBounds(bounds) and keep T01/T60 filter")
    elif n_point == 0 and n_bbox > 0:
        print("VERDICT: filterBounds(point) broken for this location")
        print("  FIX   : Use filterBounds(bounds)")
    elif n_bbox == 0:
        print("VERDICT: No imagery available — cloud threshold too strict")
        print(f"  FIX   : Increase cloud_threshold above {cloud_threshold}%")
    elif n_filtered > 0:
        print("VERDICT: No issue — real tiles found via point query")
        print("  DATA  : Should extract correctly with current pipeline")
    print()

In [62]:
if summary is not None:
    failing = summary[summary['null_%'] > 0]
    for _, row in failing.iterrows():
        sample = main_df[main_df['dataset'] == row['dataset']].iloc[0]
        diagnose_tile_coverage(
            dataset_name = row['dataset'],
            lon          = sample['longitude'],
            lat          = sample['latitude'],
            start_date   = sample['capture_start'],
            end_date     = sample['capture_end']
        )

**Coastal mangrove sites at 10% cloud threshold**  
 - All 7 fully-missing datasets are coastal mangrove regions (Chiriquí, Sierpe, Amazon delta).  
 - These are among the cloudiest biomes on earth.  
 - At 10% threshold there are simply zero qualifying images.  
 - The Brazil datasets that do work (Brazil-Salinas, Brazil-Caetano etc.) are slightly further inland or in drier coastal zones.

## CHECK IMAGE AVAILABILITY FOR DIFFERENT CLOUD THRESHOLDS

In [ ]:
def audit_image_availability(df, datasets_to_check, thresholds=[10, 20, 30, 50]):
    """
    For each failing dataset, check how many Sentinel images
    exist at each cloud threshold.
    """
    results = []

    for dataset_name in datasets_to_check:
        subset = df[df['dataset'] == dataset_name].iloc[:1]  # one representative point
        lat    = subset['latitude'].values[0]
        lon    = subset['longitude'].values[0]
        start  = subset['capture_start'].values[0]
        end    = subset['capture_end'].values[0]

        point = ee.FeatureCollection([
            ee.Feature(ee.Geometry.Point([lon, lat]))
        ])

        print(f"\nAuditing: {dataset_name} | ({lat:.3f}, {lon:.3f}) | {start} → {end}")

        for threshold in thresholds:
            col = ee.ImageCollection('NASA/HLS/HLSS30/v002')\
                    .filterDate(start, end)\
                    .filterBounds(point)\
                    .filter(ee.Filter.lt('CLOUD_COVERAGE', threshold))
            n = col.size().getInfo()
            results.append({
                'dataset'         : dataset_name,
                'lat'             : lat,
                'lon'             : lon,
                'cloud_threshold' : threshold,
                'n_images'        : n
            })
            print(f"  CLOUD_COVERAGE < {threshold:>2}%  →  {n} images")

    return pd.DataFrame(results)

In [ ]:
# Run only on the failing datasets
failing_datasets = [
    'Panama-Chirqui_2',
    'Panama-Chirqui_1',
    'Brazil-BocaGrande',
    'Brazil-FuroGrande',
    'Brazil-Furo_Do_Chato',
    'Brazil-Mangue',
    'CostaRica-Sierpe',
    'CostaRica-Nicoya',
]

availability_df = audit_image_availability(main_df, failing_datasets)

print("\n=== Summary: minimum threshold needed per dataset ===")
viable = availability_df[availability_df['n_images'] > 0]\
            .groupby('dataset')['cloud_threshold']\
            .min()\
            .reset_index()\
            .rename(columns={'cloud_threshold': 'min_threshold_with_data'})
print(viable.to_string(index=False))

## SAVE TO CSV

In [69]:
if os.path.exists(OUTPUT_CSV_MAIN_SENTINEL):
    os.remove(OUTPUT_CSV_MAIN_SENTINEL)
    print(f"Deleted old output file: {OUTPUT_CSV_MAIN_SENTINEL}")

main_df.to_csv(OUTPUT_CSV_MAIN_SENTINEL, index=False)
print(f"Saved output to: {OUTPUT_CSV_MAIN_SENTINEL}")

Saved output to: ../../DATA/AGB_DATA/Merged_Data/Sentinel_AGB/AGB_EO_SENTINEL.csv
